# Data analysis for AMC chains

In [83]:
# Import libraries I may need

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Validate scope

1. Count to ensure theater had more than 10 showings total.
2. Return a list of all movie titles and verify online they have captioned versions available.
3. Count screenings per film per theater in the week and remove films that had fewer than 4 screenings during the week; these do not fall under the rule.

In [84]:
## Import .CSV

amc_screenings = pd.read_csv('/Users/jackwalker/Data/2026/CUNY_fall/adv-data-assignment-1/data/import/screenings/raw/amc/amc.csv')
print(amc_screenings)

            Date Day of week                           Film  Showtime  \
0     2026-11-02      Monday  AMC Screen Unseen: November 2   7:00 PM   
1     2026-11-02      Monday                  La Bola Negra  11:30 AM   
2     2026-11-02      Monday                  La Bola Negra  12:10 PM   
3     2026-11-02      Monday                  La Bola Negra   3:15 PM   
4     2026-11-02      Monday                  La Bola Negra   3:45 PM   
...          ...         ...                            ...       ...   
2459  2026-11-08      Sunday            Godzilla Minus Zero   9:45 PM   
2460  2026-11-08      Sunday             The Cat in the Hat   1:45 PM   
2461  2026-11-08      Sunday             The Cat in the Hat   4:40 PM   
2462  2026-11-08      Sunday             The Cat in the Hat   7:30 PM   
2463  2026-11-08      Sunday             The Cat in the Hat  10:20 PM   

     Peak hours? Open captioned?              Theater Company  \
0            Yes              No  AMC 19th St. East 6     

In [85]:
## Check for number of screenings per theater
amc_screenings['Theater'].value_counts()

Theater
AMC Empire 25                   474
AMC Lincoln Square 13           339
AMC Kips Bay 15                 269
AMC 34th Street 14              257
AMC Bay Plaza Cinema 13         197
AMC DINE-IN Staten Island 11    192
AMC Magic Johnson Harlem 9      143
AMC Fresh Meadows 7             120
AMC Village 7                   113
AMC Bay Terrace 6               109
AMC 84th Street 6               102
AMC 19th St. East 6             101
AMC Orpheum 7                    48
Name: count, dtype: int64

In [86]:
## Okay, they all have more than 10.

## Now I'm going to store each unique film title in a .CSV file and then
## manually check online to confirm it has a captioned version possible.

open_caption_available = (
    amc_screenings[['Film']]
    .drop_duplicates()
    .sort_values('Film')
    .reset_index(drop=True)
)

open_caption_available['Available?'] = pd.Series(
    pd.NA, index=open_caption_available.index, dtype='boolean'
)

## print(open_caption_available)

## From a manual check online, some are unclear on caption access

captions_missing = [
    'AMC Screen Unseen: November 2',
    'Club Kid with Q&A',
    'I Play Rocky Special Q&A with Cast/Filmmakers',
    'Rammstein - Live in Mexico City',
    "Stray Kids World Tour 'RUN IT' LIVE VIEWING",
    'The Cat in the Hat : Private Theatre Rental',
]

# Set False for those titles, True for all others
open_caption_available['Available?'] = ~open_caption_available['Film'].isin(captions_missing)

print(open_caption_available)

                                   Film  Available?
0                                 #CULT        True
1                   A Talent for Murder        True
2         AMC Screen Unseen: November 2       False
3      Aadarsha Kutumbam - House No. 47        True
4                               Breeder        True
..                                  ...         ...
57                      Wild Horse Nine        True
58              Wild at Heart - Unrated        True
59                             Wildwood        True
60               You Can See Everything        True
61  Your Mother Your Mother Your Mother        True

[62 rows x 2 columns]


In [87]:
## Remove the ones we can't confirm from our dataset; this makes our final
## result a minimum number of screenings failing to meet the mark, as 
## opposed to an overshot bc some didn't even have captions as an option.

amc_screenings_filtered = amc_screenings[
    ~amc_screenings['Film'].isin(captions_missing)
].copy()

print('Rows removed:', len(amc_screenings) - len(amc_screenings_filtered))

Rows removed: 33


In [88]:
## Remove screenings with fewer than four per theater.

## Store counts of films per theater using groupby
screenings_count = amc_screenings_filtered.groupby(['Theater', 'Film']).size()

# Find theater screenings exempt due to frequency criteria
screenings_too_few = screenings_count[screenings_count < 4].index

## Remove those screenings
amc_screenings_filtered = (
    amc_screenings_filtered
    .set_index(['Theater', 'Film'])
    .drop(index=screenings_too_few, errors='ignore')
    .reset_index()
    .copy()
)

print(amc_screenings_filtered)

                  Theater                 Film        Date Day of week  \
0     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
1     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
2     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
3     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
4     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
...                   ...                  ...         ...         ...   
2300        AMC Village 7  Godzilla Minus Zero  2026-11-08      Sunday   
2301        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   
2302        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   
2303        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   
2304        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   

      Showtime Peak hours? Open captioned? Company  \
0     11:30 AM          No              No     AMC   
1  

## 2. Check for proportion of open captioned screenings

1. Identify the number of showings of each film at each theater in the week. Store film name, theater and count in a new dataframe using groupby.
2. Count how many of those showings were open captioned. Store the number in a new column.
3. Calculate the percentage of screenings per film per theater that were open captioned, and store the calculation in a new column.
4. Check if that percent is greater than or equal to the minimum set by state law. Create a new Adhering? column and store a boolean there based on whether or not they're adhering to state law
5. Return a new dataframe just of violations, based on where Adhering? == False
6. Print a count of violations per individual theater.

In [89]:
## Create our new dataframe

screenings_per_theater = (
    amc_screenings_filtered
    .groupby(['Theater', 'Film'])
    .size()
    .reset_index(name='Total screenings')
)

print(screenings_per_theater)

                 Theater                       Film  Total screenings
0    AMC 19th St. East 6              La Bola Negra                22
1    AMC 19th St. East 6                     Wicker                15
2    AMC 19th St. East 6            Wild Horse Nine                18
3    AMC 19th St. East 6     You Can See Everything                45
4     AMC 34th Street 14  Christmas at the Kringles                 9
..                   ...                        ...               ...
164        AMC Village 7         The Cat in the Hat                15
165        AMC Village 7          The Mosquito Bowl                12
166        AMC Village 7                     Verity                 6
167        AMC Village 7                   Wildwood                12
168        AMC Village 7     You Can See Everything                23

[169 rows x 3 columns]


In [90]:
## Add a column for open captioned totals
screenings_per_theater['Total captioned'] = (
    amc_screenings_filtered
    .assign(is_oc=amc_screenings_filtered['Open captioned?'].eq('Yes'))
    .groupby(['Theater', 'Film'])['is_oc']
    .sum()
    .values
)

print(screenings_per_theater)

                 Theater                       Film  Total screenings  \
0    AMC 19th St. East 6              La Bola Negra                22   
1    AMC 19th St. East 6                     Wicker                15   
2    AMC 19th St. East 6            Wild Horse Nine                18   
3    AMC 19th St. East 6     You Can See Everything                45   
4     AMC 34th Street 14  Christmas at the Kringles                 9   
..                   ...                        ...               ...   
164        AMC Village 7         The Cat in the Hat                15   
165        AMC Village 7          The Mosquito Bowl                12   
166        AMC Village 7                     Verity                 6   
167        AMC Village 7                   Wildwood                12   
168        AMC Village 7     You Can See Everything                23   

     Total captioned  
0                  0  
1                  1  
2                  1  
3                  1  
4       

In [91]:
## Add a column calculating proportion that are open captioned
screenings_per_theater['Captioned proportion'] = (
    screenings_per_theater['Total captioned'] / screenings_per_theater['Total screenings']
)

print(screenings_per_theater)

                 Theater                       Film  Total screenings  \
0    AMC 19th St. East 6              La Bola Negra                22   
1    AMC 19th St. East 6                     Wicker                15   
2    AMC 19th St. East 6            Wild Horse Nine                18   
3    AMC 19th St. East 6     You Can See Everything                45   
4     AMC 34th Street 14  Christmas at the Kringles                 9   
..                   ...                        ...               ...   
164        AMC Village 7         The Cat in the Hat                15   
165        AMC Village 7          The Mosquito Bowl                12   
166        AMC Village 7                     Verity                 6   
167        AMC Village 7                   Wildwood                12   
168        AMC Village 7     You Can See Everything                23   

     Total captioned  Captioned proportion  
0                  0              0.000000  
1                  1             

In [92]:
## Add a column clarifying whether that meets the state law
screenings_per_theater['Adhering?'] = (
    screenings_per_theater['Captioned proportion'] >= 0.25
)

print(screenings_per_theater)

                 Theater                       Film  Total screenings  \
0    AMC 19th St. East 6              La Bola Negra                22   
1    AMC 19th St. East 6                     Wicker                15   
2    AMC 19th St. East 6            Wild Horse Nine                18   
3    AMC 19th St. East 6     You Can See Everything                45   
4     AMC 34th Street 14  Christmas at the Kringles                 9   
..                   ...                        ...               ...   
164        AMC Village 7         The Cat in the Hat                15   
165        AMC Village 7          The Mosquito Bowl                12   
166        AMC Village 7                     Verity                 6   
167        AMC Village 7                   Wildwood                12   
168        AMC Village 7     You Can See Everything                23   

     Total captioned  Captioned proportion  Adhering?  
0                  0              0.000000      False  
1          

In [93]:
# Store violating films/theaters in a new dataframe --
# ALSO do I need to confirm that these are exhaustive lists and it won't
# be updated further?

violations = screenings_per_theater[screenings_per_theater['Adhering?'] == False].copy()
#violations['Adhering?'].value_counts()

print(violations)

                 Theater                       Film  Total screenings  \
0    AMC 19th St. East 6              La Bola Negra                22   
1    AMC 19th St. East 6                     Wicker                15   
2    AMC 19th St. East 6            Wild Horse Nine                18   
3    AMC 19th St. East 6     You Can See Everything                45   
4     AMC 34th Street 14  Christmas at the Kringles                 9   
..                   ...                        ...               ...   
163        AMC Village 7             Street Fighter                12   
164        AMC Village 7         The Cat in the Hat                15   
165        AMC Village 7          The Mosquito Bowl                12   
167        AMC Village 7                   Wildwood                12   
168        AMC Village 7     You Can See Everything                23   

     Total captioned  Captioned proportion  Adhering?  
0                  0              0.000000      False  
1          

In [94]:
amc_violation_count = len(violations)
amc_total_count = len(amc_screenings_filtered)
amc_violation_rate = (amc_violation_count / amc_total_count)

print(f"AMC total relevant screenings: {amc_total_count}.")
print(f"AMC total violations: {amc_violation_count}.")
print(f"From Nov. 2 to 8, 2026, AMC failed to meet open caption laws for {amc_violation_rate * 100:.2f}% of the film titles it screened.")

AMC total relevant screenings: 2305.
AMC total violations: 147.
From Nov. 2 to 8, 2026, AMC failed to meet open caption laws for 6.38% of the film titles it screened.


In [102]:
## Would be good to break this down further by theater. Let's print a count
## of violations per theater, plus total screenings.

theater_violation_totals = (
    violations.groupby('Theater')
    .size()
    .sort_values(ascending=False)
)

theater_violation_totals_df = theater_violation_totals.reset_index(name='Violation count')

# Filter for totals
theater_screening_totals_df = (
    amc_screenings_filtered.groupby('Theater')
    .size()
    .reset_index(name='Total screenings')
)

#Filter for violations
theater_violation_totals_df = (
    theater_violation_totals_df
    .merge(theater_screening_totals_df, on='Theater', how='left')
    .sort_values('Violation count', ascending=False)
    .reset_index(drop=True)
)

#Calculate rate of violations
theater_violation_totals_df = (
    theater_violation_totals_df
    .assign(**{
        'Violation rate (%)': (
            theater_violation_totals_df['Violation count']
            / theater_violation_totals_df['Total screenings'] * 100
        ).round(2)
    })
)

print(theater_violation_totals_df.sort_values('Violation rate (%)', ascending=False))

                         Theater  Violation count  Total screenings  \
12                 AMC Orpheum 7                4                47   
6              AMC Bay Terrace 6                9               108   
7            AMC Fresh Meadows 7                8               110   
9              AMC 84th Street 6                7                99   
4   AMC DINE-IN Staten Island 11               13               185   
1             AMC 34th Street 14               16               236   
5        AMC Bay Plaza Cinema 13               12               183   
10                 AMC Village 7                7               107   
0                  AMC Empire 25               28               431   
3                AMC Kips Bay 15               15               239   
8     AMC Magic Johnson Harlem 9                8               134   
2          AMC Lincoln Square 13               16               326   
11           AMC 19th St. East 6                4               100   

    V

## 3. Check for time of day adherence

(I am less interested in exploring this side of the data in the final piece. I think it would make it harder to scale up. This part of the law is also quite confusing, and I wonder if that plays a role in adherence from theaters.)

1. Using our filtered dataframe, let's calculate how many open captioned films fell within peak hours as set by the law. Let's store totals and rates as a new dataframe.

For reference: 

### Peak hours definition
- Friday: 5:59 PM–11:01 PM
- Saturday/Sunday: 11:29 AM–11:01 PM

### Peak allocation rule
- At least half of required OC showings in peak hours.
- Exception: if fewer than 1/8 of film showings are in peak hours, then all peak-hour showings must be open captioned.

### Weekday evening rule (outside peak)
- For required OC showings outside peak, at least half should be Mon–Thu, starting after 5:59 PM and finishing before 11:01 PM.
- Exception: if fewer than 1/8 of film showings are after 5:59 PM Mon–Thu, then all such showings must be OC.


2. Let's calculate whether at least half of the open captioned screenings fall
in peak hours. Store the value as a boolean in a new column of our new dataframe.

3. Let's calculate whether fewer than 1/8 of a film's overall screenings are in peak hours, and store that info as a boolean.

4. If the answer to the previous question is true, then check if all peak hour screenings are open captioned, and store the finding as a boolean in a new column.

5. Check how many open captioned showings are outside peak, and store that number in a new column.

7. Check how many of those are Mon–Thu, starting after 5:59 PM and finishing before 11:01 PM. Store count in a new column.

8. If that number is more than half of the outside peak showings, store a true value in a new column. If not, store a false value.

9. Calculate whether fewer than 1/8 of a film's overall screenings are after 5:59 PM Mon–Thu, , and store that info as a boolean.

10. If the answer to the previous question is true, then check if all non peak hour screenings are open captioned, and store the finding as a boolean in a new column.